# Gold fact: `fact_region_flood_exposure`

## Purpose

Publish how much of each region lies in each approved MGB flood-susceptibility level, for the flood-control comparison.

## Grain

One region per susceptibility level, MGB snapshot, and boundary version.

- Uniqueness: `region_key`, `flood_susceptibility_key`, `mgb_source_snapshot_id`, `boundary_version`.

## Primary key

`region_flood_exposure_key` is `XXHASH64` of the label `REGION_FLOOD_EXPOSURE` and those four parts. See decision [D-33](../../docs/decisions.md).

## Inputs

- `02-silver.silver_region_flood_exposure` for every measure
- `03-gold.dim_region` for `region_key`
- `03-gold.dim_flood_susceptibility` for `flood_susceptibility_key`
- `04-validation.silver_dq_results` for the Silver STOP gate

## Outputs

`03-gold.fact_region_flood_exposure`

## Not done here

- Spatial work. Silver already clipped the polygons, dissolved overlap, and measured area in an equal-area CRS in `09_silver_region_flood_exposure`. Gold runs no spatial function.
- MGB code mapping. Silver applied the approved mapping.
- Turning no data into zero. A region without a safe boundary keeps `NULL` area and share.

Flood levels can overlap. Never add level areas for a region unless `level_area_additivity_status` is `ADDITIVE_WITHIN_TOLERANCE`.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` and `MERGE` on the deterministic key. Rows are never deleted.

## 1. Confirm upstream safety and select versions

**What:** check that `silver_region_flood_exposure` is one validated run with one MGB snapshot and one boundary version, and that the Gold dimensions exist.

**Why:** Gold reads only validated Silver outputs. See [Gold model load pattern](../../docs/gold_model.md#load-pattern) and decision [D-29](../../docs/decisions.md).

**Protects:** Consistency and Timeliness. Every check stops the notebook before any write.

**Expected result:** every `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

-- gold rule label. it changes the gold run id when this notebook's logic changes (d-29, d-33)
DECLARE OR REPLACE VARIABLE exposure_rule_version STRING DEFAULT 'gold-fact-region-flood-exposure-v1';
-- source system of the approved mgb mapping. read from dim_flood_susceptibility below, so 02 sets it once (d-30)
DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT mgb_source_snapshot_id) = 1
    AND COUNT(DISTINCT boundary_version) = 1
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'region', psgc_region_code, 'level', flood_susceptibility_level
    )),
    'STOP: silver_region_flood_exposure must be nonempty, one run, and unique per region and level'
)
FROM `02-silver`.silver_region_flood_exposure;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver regional flood-exposure validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_region_flood_exposure')
  AND dq.pipeline_run_id = (SELECT MAX(exposure.run_id) FROM `02-silver`.silver_region_flood_exposure AS exposure);

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `03-gold`.dim_region WHERE region_key = 0) = 1
    AND (SELECT COUNT(*) FROM `03-gold`.dim_flood_susceptibility WHERE flood_susceptibility_key = 0) = 1,
    'STOP: run the Gold dimension notebooks 00 to 04 before the facts'
);

SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_system) = 1,
    'STOP: dim_flood_susceptibility must hold exactly one MGB source system'
)
FROM `03-gold`.dim_flood_susceptibility;

SET VAR mgb_mapping_source_system = (
    SELECT MAX(source_system) FROM `03-gold`.dim_flood_susceptibility
);

## 2. Create the table

**What:** create `03-gold.fact_region_flood_exposure` with the column names, types, and order in `docs/gold_model.md`.

**Why:** [Gold model](../../docs/gold_model.md#fact_region_flood_exposure) is the column contract. Genie reads every comment.

**Protects:** Validity. `tests/test_gold_facts.py` fails if a name, type, or order drifts.

**Expected result:** the table exists. A rerun changes nothing.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.fact_region_flood_exposure (
    region_flood_exposure_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the label REGION_FLOOD_EXPOSURE, the region key, flood level key, MGB snapshot, and boundary version.',
    region_key BIGINT NOT NULL COMMENT 'Region of this exposure. Joins to dim_region.region_key.',
    flood_susceptibility_key INT NOT NULL COMMENT 'Approved MGB flood level. Joins to dim_flood_susceptibility.flood_susceptibility_key.',
    susceptible_area_sqkm DECIMAL(18,4) COMMENT 'Region area inside this flood level, in square kilometres from an equal-area CRS. NULL when the region has no safe boundary. Zero is real zero.',
    share_of_region_area_pct DECIMAL(7,4) COMMENT 'Percent of the region area inside this flood level, from 0 to 100. Non-additive. Do not sum it.',
    source_polygon_count BIGINT COMMENT 'Number of MGB source polygons that contributed to this region and level.',
    mgb_source_version STRING COMMENT 'MGB publisher version. NULL because the approved extract has none. See mgb_source_snapshot_id.',
    boundary_version STRING COMMENT 'Publisher version of the region boundaries used for clipping.',
    spatial_match_status STRING COMMENT 'CALCULATED, NO_MAPPED_EXPOSURE for a safe region with zero exposure, or NO_SAFE_REGION_BOUNDARY for no data.',
    area_calculation_crs STRING COMMENT 'Equal-area coordinate reference system used for the area.',
    area_rule_version STRING COMMENT 'Silver area rule version.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver exposure run and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Latest Bronze load time of the MGB and boundary snapshots behind this row.',
    level_area_additivity_status STRING COMMENT 'Add level areas for one region only when this is ADDITIVE_WITHIN_TOLERANCE. Flood levels can overlap.',
    mgb_source_snapshot_id STRING COMMENT 'Selected MGB snapshot. Keeps the grain traceable because the MGB publisher version is NULL.'
)
USING DELTA
COMMENT 'Gold regional flood-exposure fact. One official region per approved MGB flood level, MGB snapshot, and boundary version. NULL area means no data, not zero exposure.';

## 3. Build and merge the rows

**What:** attach both keys from the Gold dimensions, check the rows, and merge by key.

**Why:**

- `region_key` joins on the PSGC code and version.
- `flood_susceptibility_key` joins on the level and the mapping source contract that Silver used.
- Gold areas must equal Silver exactly, so the check compares the area total and the `NULL` count before the write.

**Protects:** Completeness, Consistency, Validity, and Uniqueness. No Silver row is dropped, no row uses key `0`, and no-data areas stay `NULL`.

**Expected result:** every Silver row resolves to a real region and a real level, keys are unique, and the area total and `NULL` count equal Silver.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW fact_region_flood_exposure_source AS
WITH resolved AS (
    SELECT
        region.region_key,
        flood.flood_susceptibility_key,
        exposure.susceptible_area_sqkm,
        exposure.share_of_region_area_pct,
        exposure.source_polygon_count,
        exposure.mgb_source_version,
        exposure.boundary_version,
        exposure.spatial_match_status,
        exposure.area_calculation_crs,
        exposure.area_rule_version,
        SHA2(CONCAT_WS('|', exposure.run_id, exposure_rule_version), 256) AS run_id,
        exposure.source_load_ts,
        exposure.level_area_additivity_status,
        exposure.mgb_source_snapshot_id
    FROM `02-silver`.silver_region_flood_exposure AS exposure
    LEFT JOIN `03-gold`.dim_region AS region
        ON exposure.psgc_region_code = region.psgc_region_code
       AND exposure.psgc_version <=> region.psgc_version
    LEFT JOIN `03-gold`.dim_flood_susceptibility AS flood
        ON exposure.flood_susceptibility_level = flood.flood_susceptibility_level
       AND exposure.mgb_mapping_source_version = flood.source_version
       AND flood.source_system = mgb_mapping_source_system
)
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'REGION_FLOOD_EXPOSURE', CAST(region_key AS STRING),
        CAST(flood_susceptibility_key AS STRING), mgb_source_snapshot_id, boundary_version
    )) AS region_flood_exposure_key,
    *
FROM resolved;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_region_flood_exposure)
    AND COUNT_IF(region_key IS NULL OR region_key = 0) = 0
    AND COUNT_IF(flood_susceptibility_key IS NULL OR flood_susceptibility_key = 0) = 0
    AND COUNT(*) = COUNT(DISTINCT region_flood_exposure_key)
    AND SUM(susceptible_area_sqkm) <=> (
        SELECT SUM(silver.susceptible_area_sqkm) FROM `02-silver`.silver_region_flood_exposure AS silver
    )
    AND COUNT_IF(susceptible_area_sqkm IS NULL) = (
        SELECT COUNT_IF(silver.susceptible_area_sqkm IS NULL) FROM `02-silver`.silver_region_flood_exposure AS silver
    ),
    'STOP: every exposure row must resolve to a real region and flood level with a unique key and the Silver areas'
)
FROM fact_region_flood_exposure_source;

MERGE INTO `03-gold`.fact_region_flood_exposure AS target
USING fact_region_flood_exposure_source AS source
ON target.region_flood_exposure_key = source.region_flood_exposure_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

# Summary

- **Built:** `03-gold.fact_region_flood_exposure`, one row per region, approved flood level, MGB snapshot, and boundary version. Areas come unchanged from Silver.
- **Coverage:** every row uses a real region and a real level. The query below shows the rows by spatial status, with the rows that have no area. Regions without a safe boundary keep `NULL` area. That means no data, not zero exposure.
- **Known limits:** flood levels can overlap. Add level areas for a region only when `level_area_additivity_status` is `ADDITIVE_WITHIN_TOLERANCE`. The MGB publisher version is `NULL`, so `mgb_source_snapshot_id` keeps the grain traceable. Flood exposure is only a proxy for need.
- **Next notebook:** `07_gold_fact_project_snapshot`.

In [ ]:
%sql
SELECT
    spatial_match_status,
    level_area_additivity_status,
    COUNT(*) AS exposure_rows,
    COUNT_IF(susceptible_area_sqkm IS NULL) AS rows_without_area
FROM `03-gold`.fact_region_flood_exposure
GROUP BY ALL
ORDER BY exposure_rows DESC;